In [1]:
# Kiểm tra Gpu
!nvidia-smi

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mon Dec 22 07:45:39 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

#Code


In [2]:
# Cài đặt dependencies
!pip install ultralytics supervision
!pip install supervision[assets]==0.24.0


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.4/212.4 kB 19.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.2/158.2 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 8.5 MB/s eta 0:00:00
  Attempting uninstall: tqdm
    Found existing installation: tqdm 4.67.1
    Uninstalling tqdm-4.67.1:
      Successfully uninstalled tqdm-4.67.1
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: supervision
    Found existing installation: supervision 0.27.0
    Uninstalling supervision-0.27.0:
      Successfully uninstalled supervision-0.27.0
ERROR: pip's dependency resolver does not

In [3]:
#Import thư viện và load video, model
import os
HOME = os.getcwd()
print(HOME)

!yolo settings sync=False

from IPython import display
display.clear_output()

import ultralytics
ultralytics.checks()

import supervision as sv
print("supervision.__version__:", sv.__version__)

import cv2
import numpy as np
import supervision as sv
from ultralytics import YOLO
from supervision.assets import VideoAssets, download_assets
from collections import defaultdict
from typing import Dict, List, Tuple, Optional
from datetime import datetime
import time

#Đường dẫn tới video (Thay đổi nếu cần)
SOURCE_VIDEO_PATH = "/content/drive/MyDrive/422001503103_Nhom_05_Theo dõi đối tượng trong video (Object Tracking)/Video/136554-764417387_small.mp4"

#Đường dẫn tới mô hình (Thay đổi nếu cần - load file best.pt)
model = YOLO("/content/drive/MyDrive/422001503103_Nhom_05_Theo dõi đối tượng trong video (Object Tracking)/Model/finetune_car_detection/yolov8_car_optimized/weights/best.pt")

Ultralytics 8.3.240 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 38.5/112.6 GB disk)
supervision.__version__: 0.24.0


In [4]:
#Code xử lý trên video

# Class selection
CLASS_NAMES_DICT = model.model.names
SELECTED_CLASS_NAMES = ['bus', 'truck', 'motorbike', 'car']
SELECTED_CLASS_IDS = [
    {value: key for key, value in CLASS_NAMES_DICT.items()}[class_name]
    for class_name in SELECTED_CLASS_NAMES
]

# Configuration
LINE_START = sv.Point(50, 700)
LINE_END = sv.Point(1870, 700)
TARGET_VIDEO_PATH = f"{HOME}/result.mp4"

# Source and Target ROIs
SOURCE = np.array([
    [600, 400],
    [1320, 400],
    [1920, 1080],
    [0, 1080],
])

TARGET_WIDTH = 125
TARGET_HEIGHT = 1500
TARGET = np.array([
    [0, 0],
    [TARGET_WIDTH - 1, 0],
    [TARGET_WIDTH - 1, TARGET_HEIGHT - 1],
    [0, TARGET_HEIGHT - 1],
])

# View Transformer
class ViewTransformer:
    def __init__(self, source: np.ndarray, target: np.ndarray) -> None:
        source = source.astype(np.float32)
        target = target.astype(np.float32)
        self.m = cv2.getPerspectiveTransform(source, target)

    def transform_points(self, points: np.ndarray) -> np.ndarray:
        if points.size == 0:
            return points
        reshaped_points = points.reshape(-1, 1, 2).astype(np.float32)
        transformed_points = cv2.perspectiveTransform(reshaped_points, self.m)
        return transformed_points.reshape(-1, 2)

view_transformer = ViewTransformer(source=SOURCE, target=TARGET)

# Video Info
video_info = sv.VideoInfo.from_video_path(SOURCE_VIDEO_PATH)
print(video_info)

# Initialize components
byte_tracker = sv.ByteTrack(
    track_activation_threshold=0.25,
    lost_track_buffer=30,
    minimum_matching_threshold=0.8,
    frame_rate=video_info.fps,
    minimum_consecutive_frames=3
)
byte_tracker.reset()

line_zone = sv.LineZone(start=LINE_START, end=LINE_END)
box_annotator = sv.BoxAnnotator(thickness=2)
label_annotator = sv.LabelAnnotator(text_thickness=1, text_scale=0.8, text_color=sv.Color.BLACK)
trace_annotator = sv.TraceAnnotator(thickness=2, trace_length=50)
line_zone_annotator = sv.LineZoneAnnotator(thickness=2, text_thickness=2, text_scale=2)

# Custom Speed Estimator - Fixed IndexError
class CustomSpeedEstimator:
    def __init__(self, view_transformer: ViewTransformer, pixel_to_meter: float = 0.05, fps: int = 24):
        self.view_transformer = view_transformer
        self.pixel_to_meter = pixel_to_meter
        self.fps = fps
        self.track_positions: Dict[int, List[Tuple[float, int]]] = defaultdict(list)
        self.time_window_frames = 5 * fps  # 5 seconds

    def tick(self, detections: sv.Detections, frame_index: int):
        # Clean old data
        for tid in list(self.track_positions.keys()):
            self.track_positions[tid] = [
                (y, f) for y, f in self.track_positions[tid]
                if frame_index - f <= self.time_window_frames
            ]
            if not self.track_positions[tid]:
                del self.track_positions[tid]

        # Update current positions
        if len(detections) == 0:
            return

        for i, tracker_id in enumerate(detections.tracker_id):
            x1, y1, x2, y2 = detections.xyxy[i]
            center_x = (x1 + x2) / 2
            bottom_y = y2
            orig_point = np.array([[center_x, bottom_y]])
            trans_point = self.view_transformer.transform_points(orig_point)
            if trans_point.size == 2:
                y_pos = trans_point[0, 1]
                self.track_positions[tracker_id].append((y_pos, frame_index))

    def get_speed_for_tracker(self, tracker_id: int) -> Optional[float]:
        positions = self.track_positions.get(tracker_id, [])
        if len(positions) < 2:
            return None
        y1, t1 = positions[0]
        y2, t2 = positions[-1]
        if t2 == t1:
            return None
        dist_pixels = abs(y2 - y1)
        dist_meters = dist_pixels * self.pixel_to_meter
        time_secs = (t2 - t1) / self.fps
        if time_secs <= 0:
            return None
        speed_ms = dist_meters / time_secs
        return speed_ms * 3.6

    def get_speeds(self, tracker_ids: List[int]) -> List[Optional[float]]:
        return [self.get_speed_for_tracker(tid) for tid in tracker_ids]

speed_estimator = CustomSpeedEstimator(
    view_transformer=view_transformer,
    pixel_to_meter=0.05,
    fps=video_info.fps
)

start_time = time.time()
fps_counter = 0
current_frame_index = 0

def draw_text_with_bg(frame, text, position, font=cv2.FONT_HERSHEY_SIMPLEX,
                      font_scale=0.8, text_color=(255, 255, 255), bg_color=(0, 0, 0), thickness=2):
    text_size = cv2.getTextSize(text, font, font_scale, thickness)[0]
    text_x, text_y = position
    bg_x1 = text_x - 5
    bg_y1 = text_y - text_size[1] - 5
    bg_x2 = text_x + text_size[0] + 5
    bg_y2 = text_y + 5
    cv2.rectangle(frame, (bg_x1, bg_y1), (bg_x2, bg_y2), bg_color, -1)
    cv2.putText(frame, text, (text_x, text_y), font, font_scale, text_color, thickness, cv2.LINE_AA)

# Callback function
def callback(frame: np.ndarray, index: int) -> np.ndarray:
    global fps_counter, start_time, current_frame_index, video_info

    current_frame_index = index + 1  # +1 because index starts from 0

    # Calculate FPS
    fps_counter += 1
    elapsed_time = time.time() - start_time
    if elapsed_time > 1.0:
        current_fps = fps_counter / elapsed_time
        fps_counter = 0
        start_time = time.time()
    else:
        current_fps = fps_counter / max(elapsed_time, 0.001)

    # Model inference and tracking
    results = model(frame, verbose=False)[0]
    detections = sv.Detections.from_ultralytics(results)
    detections = detections[np.isin(detections.class_id, SELECTED_CLASS_IDS)]
    detections = byte_tracker.update_with_detections(detections)

    # Update speed estimator
    speed_estimator.tick(detections=detections, frame_index=index)
    speeds = speed_estimator.get_speeds(detections.tracker_id) if len(detections) > 0 else []

    # Build labels with speed
    labels = []
    for i, (conf, cls_id, tracker_id) in enumerate(zip(detections.confidence, detections.class_id, detections.tracker_id)):
        label = f"#{tracker_id} {model.model.names[cls_id]} {conf:.2f}"
        if i < len(speeds) and speeds[i] is not None:
            label += f" {speeds[i]:.1f} km/h"
        labels.append(label)

    # Annotate frame
    annotated_frame = frame.copy()
    annotated_frame = trace_annotator.annotate(scene=annotated_frame, detections=detections)
    annotated_frame = box_annotator.annotate(scene=annotated_frame, detections=detections)
    annotated_frame = label_annotator.annotate(scene=annotated_frame, detections=detections, labels=labels)

    # Line counter
    line_zone.trigger(detections=detections)
    annotated_frame = line_zone_annotator.annotate(annotated_frame, line_counter=line_zone)

    # === Add FPS + Frame Counter in Top-Right ===
    fps_text = f"FPS: {current_fps:.1f}"
    frame_text = f"{current_frame_index}/{video_info.total_frames}"

    # Position: top-right corner with some padding
    fps_pos = (annotated_frame.shape[1] - 350, 40)
    frame_pos = (annotated_frame.shape[1] - 150, 40)

    draw_text_with_bg(annotated_frame, fps_text, fps_pos, font_scale=0.9, thickness=2)
    draw_text_with_bg(annotated_frame, frame_text, frame_pos, font_scale=0.9, thickness=2)

    return annotated_frame

# Process video
sv.process_video(
    source_path=SOURCE_VIDEO_PATH,
    target_path=TARGET_VIDEO_PATH,
    callback=callback
)


VideoInfo(width=540, height=960, fps=25, total_frames=314)


In [ ]:
#Code để tải video đã qua xử lý về máy
from google.colab import files

if os.path.exists(TARGET_VIDEO_PATH):
    files.download(TARGET_VIDEO_PATH)
else:
    print(f"File not found at {TARGET_VIDEO_PATH}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>